In [30]:
pip install PyPDF2


Note: you may need to restart the kernel to use updated packages.


In [31]:
!pip install faiss-cpu

In [45]:
import PyPDF2
import faiss
import numpy as np
from openai import OpenAI
import os

client = OpenAI(
    api_key="JFuVvJ5WMJ1OQWshT3BlbkFJ9jh-Hk1fFYQ2Qh_jqpvxarA6MB03CbBednstxGwp7sT_q2Ml7t0sJeTZIj2TNIrzVkysQH_kUA"
)


def read_pdf(path):
    reader = PyPDF2.PdfReader(path)
    text = ""

    for page in reader.pages:
        page_text = page.extract_text()

        if page_text:
            text = text + page_text

    return text


def make_chunks(text, chunk_size=500):
    chunks = []
    start = 0

    while start < len(text):
        end = start + chunk_size
        chunk = text[start:end]
        chunks.append(chunk)
        start = end

    return chunks


def get_embeddings(chunklist):
    embeddings = []

    for chunk in chunklist:
        response = client.embeddings.create(
            input=chunk,
            model="text-embedding-3-small"
        )

        vector = response.data[0].embedding
        embeddings.append(vector)

    return np.array(embeddings).astype("float32")


def build_faiss_index(embeddings):
    dimensions = len(embeddings[0])

    index = faiss.IndexFlatL2(dimensions)

    index.add(embeddings)

    return index


def search_chunks(question, chunks, index):

    q_embed = client.embeddings.create(
        input=question,
        model="text-embedding-3-small"
    ).data[0].embedding

    q_embed = np.array([q_embed]).astype("float32")

    distances, indices = index.search(q_embed, k=2)

    best_chunks = [
        chunks[i]
        for i in indices[0]
    ]

    return best_chunks


def ask_question(chunks, question):

    context = "\n\n".join(chunks)

    prompt = f"""
Context:
{context}

Question:
{question}

Answer the question using the context provided.
"""

    response = client.chat.completions.create(
        model="gpt-4o-mini",
        messages=[
            {
                "role": "user",
                "content": prompt
            }
        ]
    )

    return response.choices[0].message.content


def main():

    text = read_pdf("Onepiece.pdf")
    print("PDF Loaded.")

    chunks = make_chunks(text)
    print("Text divided into chunks.")

    embeddings = get_embeddings(chunks)
    print("Embeddings created.")

    index = build_faiss_index(embeddings)
    print("FAISS index created.")

    question = input("Ask your Question from PDF: ")

    best_chunks = search_chunks(
        question,
        chunks,
        index
    )

    print("Found related text parts.")

    print(best_chunks)

    print("***********************************")

    answer = ask_question(
        best_chunks,
        question
    )

    print("\nAnswer from OpenAI:")
    print(answer)


if __name__ == "__main__":
    main()

PDF Loaded.
Text divided into chunks.
Embeddings created.
FAISS index created.


Ask your Question from PDF:  who is the father of luffy


Found related text parts.
['he captain of the Straw Hat Pirates. He is energetic, optimistic, loyal, and\ndetermined. His greatest goal is to find the legendary treasure known as the One Piece and become the\nPirate King. Luffy is famous for his unusual fighting style, strong will, and ability to inspire people around\nhim. His journey is not only about becoming powerful; it is also about friendship, freedom, and protecting\npeople who are important to him.\nThe World of One Piece\nThe world of One Piece is divided into many sea', "y. Franky is a shipwright who wants to\nbuild a ship capable of reaching the end of the Grand Line. Brook is a musician and swordsman who\nwants to reunite with Laboon. Jinbe is a fish-man and experienced helmsman who becomes an\nimportant member of the crew.\nDevil Fruits and Haki\nOne Piece includes several unusual power systems. Devil Fruits are mysterious fruits that can give\npeople special abilities after they eat them. These powers can be extremely va